In [0]:
hr_bronze = spark.table(
    "health_and_habit_guardian.bronze.heart_rate"
)

In [0]:
from pyspark.sql import functions as F

# explicitly cast types in the Silver layer to estabilish a canonical schema

hr = (
    hr_bronze
    .withColumn(
        "heart_rate_bpm",
        F.col("heart_rate_bpm").cast("int")
    )
)

In [0]:
from pyspark.sql.window import Window

# heart-rate deduplication

window = Window.partitionBy(
    "user_id",
    "timestamp",
    "source"
).orderBy(
    F.col("_ingested_at").desc()
)

In [0]:
# adding a quality status

hr = (
    hr
    .withColumn(
        "_quality_status",
        F.when(
            F.col("user_id").isNull() |
            F.col("timestamp").isNull() |
            (F.col("heart_rate_bpm") < 30) |
            (F.col("heart_rate_bpm") > 250),
            F.lit("INVALID")
        ).otherwise(F.lit("VALID"))
    )
)

In [0]:
hr = hr.withColumn(
    "_processed_at",
    F.current_timestamp()
)

In [0]:
# validate activity_states

valid_states = [
    "resting",
    "walking",
    "running",
    "cycling",
    "sleeping",
    "unknown"
]

hr = hr.withColumn(
    "_invalid_activity_state",
    ~F.col("activity_state").isin(valid_states)
)

In [0]:
valid_hr = (
    hr
    .filter(F.col("_quality_status") == "VALID")
)

invalid_hr = (
    hr
    .filter(F.col("_quality_status") == "INVALID")
)

In [0]:
(
    invalid_hr
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "health_and_habit_guardian.silver.quarantine_heart_rate"
    )
)

In [0]:
(
    valid_hr
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "health_and_habit_guardian.silver.heart_rate"
    )
)